# Otras Tareas de Vision: Showcase de Modelos Fundacionales

Este notebook es un showcase de otras aplicaciones de vision por computador que puedes hacer con modelos fundacionales. Son ejemplos cortos para que veas las posibilidades y te inspires para tus propios proyectos.

**Tareas que veremos:**
- OCR: Extraer texto de imagenes
- Superresolucion: Mejorar calidad de imagenes
- Background Removal: Eliminar fondos
- Depth Estimation: Calcular profundidad

Todos usando modelos de Hugging Face, listos para usar sin entrenar.

## Configuracion e Imports

In [ ]:
import torch
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Swin2SRForImageSuperResolution, AutoImageProcessor as Swin2SRProcessor
from transformers import pipeline
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## 1. OCR: Reconocimiento de Texto

Extraer texto de imagenes es util para digitalizar documentos, leer etiquetas industriales, procesar facturas, etc.

Usaremos **TrOCR**, un modelo de Microsoft que combina Vision Transformer (ViT) con GPT para leer texto.

In [ ]:
# Cargar modelo TrOCR
ocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
ocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed").to(device)

print("Modelo OCR cargado")

In [ ]:
def extract_text(image):
    """Extrae texto de imagen usando TrOCR"""
    
    # Procesar imagen
    pixel_values = ocr_processor(image, return_tensors="pt").pixel_values.to(device)
    
    # Generar texto
    with torch.no_grad():
        generated_ids = ocr_model.generate(pixel_values)
    
    text = ocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    print(f"Texto extraido: {text}")
    
    # Visualizar
    plt.imshow(image)
    plt.title(f"Texto: {text}")
    plt.axis("off")
    plt.show()
    
    return text

# Ejemplo: imagen con texto simple
# Nota: TrOCR funciona mejor con imagenes de texto recortadas (una linea)
# Para este ejemplo, necesitarias una imagen de texto. Puedes crear una simple:

# Crear imagen de texto de ejemplo
img_text = Image.new('RGB', (300, 60), color='white')
from PIL import ImageDraw, ImageFont
draw = ImageDraw.Draw(img_text)
draw.text((10, 20), "Hello World 2025", fill='black')

text = extract_text(img_text)

**Nota sobre OCR**: Para OCR mas robusto en imagenes complejas, considera usar **EasyOCR** o **PaddleOCR** que detectan y leen multiples lineas automaticamente.

## 2. Superresolucion: Mejora de Calidad

La superresolucion aumenta la resolucion de imagenes de baja calidad. Util para:
- Mejorar imagenes antiguas o comprimidas
- Zoom digital sin perdida de calidad
- Restauracion de imagenes

Usaremos **Swin2SR**, un modelo basado en Swin Transformer.

In [ ]:
# Cargar modelo Swin2SR (x2 upscaling)
sr_processor = Swin2SRProcessor.from_pretrained("caidas/swin2SR-classical-sr-x2-64")
sr_model = Swin2SRForImageSuperResolution.from_pretrained("caidas/swin2SR-classical-sr-x2-64").to(device)

print("Modelo Superresolucion cargado")

In [ ]:
def upscale_image(image):
    """Aumenta resolucion de imagen usando Swin2SR"""
    
    # Procesar imagen
    inputs = sr_processor(image, return_tensors="pt").to(device)
    
    # Aplicar superresolucion
    with torch.no_grad():
        outputs = sr_model(**inputs)
    
    # Post-procesar
    output = outputs.reconstruction.squeeze().cpu().clamp(0, 1).numpy()
    output = np.transpose(output, (1, 2, 0))  # CHW -> HWC
    output = (output * 255).astype(np.uint8)
    
    output_img = Image.fromarray(output)
    
    # Visualizar comparacion
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(image)
    axes[0].set_title(f"Original {image.size[0]}x{image.size[1]}")
    axes[0].axis("off")
    
    axes[1].imshow(output_img)
    axes[1].set_title(f"Upscaled {output_img.size[0]}x{output_img.size[1]}")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return output_img

# Ejemplo: cargar imagen de baja resolucion
url_img = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/fruits.jpg"
img_low_res = Image.open(BytesIO(requests.get(url_img).content)).convert("RGB")

# Reducir resolucion primero para demostrar
w, h = img_low_res.size
img_low_res = img_low_res.resize((w//2, h//2), Image.BILINEAR)

# Aplicar superresolucion
img_high_res = upscale_image(img_low_res)

## 3. Background Removal: Eliminacion de Fondo

Eliminar el fondo de imagenes es util para:
- Ecommerce: aislar productos para catalogos
- Fotografia: cambiar fondos
- Videos: efectos especiales

Usaremos **RMBG (Remove Background)** de BRIA AI.

In [ ]:
# Cargar pipeline de background removal
bg_remover = pipeline(
    "image-segmentation",
    model="briaai/RMBG-1.4",
    trust_remote_code=True,
    device=0 if torch.cuda.is_available() else -1
)

print("Modelo Background Removal cargado")

In [ ]:
def remove_background(image):
    """Elimina fondo de imagen"""
    
    # Aplicar remocion de fondo
    result = bg_remover(image)
    
    # El resultado es una mascara
    mask = result[0]['mask']
    
    # Aplicar mascara a imagen original
    img_array = np.array(image)
    mask_array = np.array(mask.convert('L'))
    
    # Crear imagen con fondo transparente
    img_rgba = np.dstack([img_array, mask_array])
    img_no_bg = Image.fromarray(img_rgba, 'RGBA')
    
    # Visualizar
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Mascara")
    axes[1].axis("off")
    
    axes[2].imshow(img_no_bg)
    axes[2].set_title("Sin Fondo")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return img_no_bg

# Ejemplo: eliminar fondo de persona
url_person = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/person_cars.jpg"
img_person = Image.open(BytesIO(requests.get(url_person).content)).convert("RGB")

img_no_bg = remove_background(img_person)

## 4. Depth Estimation: Estimacion de Profundidad

Calcular la profundidad de cada pixel en una imagen permite:
- Reconstruccion 3D
- Navegacion de robots
- Realidad aumentada
- Efectos de desenfoque selectivo

Usaremos **Depth Anything V2**, un modelo SOTA de Meta AI.

In [ ]:
# Cargar pipeline de depth estimation
depth_estimator = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if torch.cuda.is_available() else -1
)

print("Modelo Depth Estimation cargado")

In [ ]:
def estimate_depth(image):
    """Estima mapa de profundidad de imagen"""
    
    # Aplicar depth estimation
    result = depth_estimator(image)
    
    depth_map = result['depth']
    depth_array = np.array(depth_map)
    
    # Normalizar para visualizacion
    depth_normalized = (depth_array - depth_array.min()) / (depth_array.max() - depth_array.min())
    
    # Visualizar con diferentes colormaps
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")
    
    im1 = axes[0, 1].imshow(depth_normalized, cmap='viridis')
    axes[0, 1].set_title("Depth Map (Viridis)")
    axes[0, 1].axis("off")
    plt.colorbar(im1, ax=axes[0, 1], label="Profundidad")
    
    im2 = axes[1, 0].imshow(depth_normalized, cmap='plasma')
    axes[1, 0].set_title("Depth Map (Plasma)")
    axes[1, 0].axis("off")
    plt.colorbar(im2, ax=axes[1, 0], label="Profundidad")
    
    im3 = axes[1, 1].imshow(depth_normalized, cmap='jet')
    axes[1, 1].set_title("Depth Map (Jet)")
    axes[1, 1].axis("off")
    plt.colorbar(im3, ax=axes[1, 1], label="Profundidad")
    
    plt.tight_layout()
    plt.show()
    
    return depth_normalized

# Ejemplo: estimar profundidad
url_scene = "https://github.com/sergiovillanueva/prompt_to_mask/raw/master/assets/cars.jpg"
img_scene = Image.open(BytesIO(requests.get(url_scene).content)).convert("RGB")

depth = estimate_depth(img_scene)

## Ejercicio: Experimenta con tus imagenes

Elige una de las tareas que hemos visto y pruebala con tus propias imagenes:

1. **OCR**: Saca foto de un texto impreso
2. **Superresolucion**: Mejora una foto antigua de baja calidad
3. **Background Removal**: Aislate del fondo en una foto tuya
4. **Depth**: Visualiza la profundidad de una escena

In [ ]:
# EJERCICIO: Prueba con tus imagenes

# Carga tu imagen
# tu_imagen = Image.open("ruta/a/tu/imagen.jpg").convert("RGB")

# Elige una tarea:
# extract_text(tu_imagen)
# upscale_image(tu_imagen)
# remove_background(tu_imagen)
# estimate_depth(tu_imagen)


## Ejercicio Extra (Comodin): Combina dos tareas

Crea un pipeline que combine dos de estas tareas. Por ejemplo:
- Eliminar fondo + superponer en imagen con depth
- OCR + clasificar tipo de documento
- Superresolucion + eliminar fondo

Se creativo!

In [ ]:
# EJERCICIO EXTRA: Pipeline combinado
def my_custom_pipeline(image):
    """Tu pipeline personalizado"""
    # Tu codigo aqui...
    pass

# Prueba:
# my_custom_pipeline(img_person)


## Mas Tareas para Explorar

Hay muchas mas tareas que puedes explorar en Hugging Face:

**Vision:**
- Image-to-Image Translation (estilo, dia a noche, etc.)
- Video Classification
- Video Segmentation
- Zero-Shot Image Classification
- Unconditional Image Generation (generar imagenes desde cero)

**Vision + Texto:**
- Visual Question Answering (VQA)
- Document Question Answering
- Image Captioning
- Text-to-Image (Stable Diffusion, DALL-E)

**Especializadas:**
- Medical Imaging (rayos X, resonancias)
- Satellite Imagery (analisis de satelites)
- Industrial Inspection (defectos, anomalias)

**Explora**: https://huggingface.co/tasks

## Resumen del Curso Completo

A lo largo de estos notebooks hemos visto:

### Viernes:
✅ **Notebook 1**: Deteccion de objetos con RF-DETR  
✅ **Notebook 2**: Introduccion a Hugging Face Hub  
✅ **Notebook 3**: Modelos multimodales (CLIP, BLIP, Grounding DINO, Qwen2.5-VL)  

### Sabado:
✅ **Notebook 4**: DINO v3 y aprendizaje auto-supervisado  
✅ **Notebook 5**: SAM2 y pipelines de segmentacion  
✅ **Notebook 6**: Estimacion de pose humana  
✅ **Notebook 7**: Showcase de tareas extras (OCR, SR, BG removal, Depth)  

### Conceptos Clave:
- Modelos fundacionales: entrenados en datos masivos, aplicables a multiples tareas
- Zero-shot learning: usar modelos sin reentrenar
- Multimodalidad: combinar vision y lenguaje
- Pipelines: combinar modelos para tareas complejas
- Transformers: arquitectura dominante en IA moderna

### Donde Seguir:
- **Hugging Face Course**: https://huggingface.co/learn
- **Papers with Code**: https://paperswithcode.com
- **Awesome Computer Vision**: https://github.com/jbhuang0604/awesome-computer-vision

**Contacto**: Si tienes dudas puedes escribirme un email!

**Gracias por participar en el curso!** 🚀